# Book Review Sentiment Analysis Test
## Testing Amazon Food Review Model on Book Reviews

**Goal**: Evaluate how well your food review sentiment model transfers to book review domain

**Expected Challenges**:
- Different vocabulary (food: 'delicious', 'tasty' vs books: 'engaging', 'compelling')
- Different review patterns
- TF-IDF weights trained on food-specific terms

---

## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns

# NLP libraries
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

print("All imports successful!")

ModuleNotFoundError: No module named 'bs4'

## 1. Create Sample Book Reviews

Since web scraping may be blocked, we'll use realistic sample data.

In [ ]:
# Sample book reviews (realistic examples)
sample_reviews = [
    # Positive reviews (Rating 4-5)
    ("This book was absolutely amazing! The plot kept me engaged from start to finish. The character development was superb and I couldn't put it down. Highly recommend to anyone who loves a good thriller!", 5),
    ("A masterpiece of modern literature. The author's writing style is captivating and beautiful. The themes explored are deep and thought-provoking. This book will stay with me for a long time.", 5),
    ("Couldn't put it down! Every chapter left me wanting more. The characters felt real and their struggles were relatable. The ending was perfect - satisfying yet surprising.", 5),
    ("One of the best books I've read this year. The pacing was excellent, never a dull moment. The world-building was incredible and I felt completely immersed.", 5),
    ("Really enjoyed this read. Great character development and interesting plot twists that kept me guessing. Well-written and engaging throughout.", 4),
    ("A good book overall. The story was compelling and the writing was solid. Some parts dragged a bit but overall a very enjoyable read.", 4),
    ("Fantastic story with memorable characters. The author did an excellent job creating tension and suspense. Would definitely read more from this author.", 4),
    ("Loved the creative premise and unique storytelling approach. The narrative was fresh and original. A delightful reading experience.", 4),
    
    # Negative reviews (Rating 1-2)
    ("Disappointing. The plot was predictable and the characters felt flat and uninteresting. I expected much more based on the reviews.", 2),
    ("I couldn't finish this book. The writing was poor and the story didn't engage me at all. The pacing was terrible and nothing happened for the first 100 pages.", 1),
    ("Not what I expected. The book dragged on endlessly and I found myself bored halfway through. The characters were unlikable and poorly developed.", 2),
    ("Weak storyline and underdeveloped characters. The plot holes were numerous and distracting. Would not recommend to others.", 2),
    ("The author tried too hard to be clever. The result was confusing and frustrating to read. The story made no sense and the ending was terrible.", 1),
    ("Terrible book. Complete waste of time and money. Poorly written with boring characters and a nonsensical plot. Save yourself the trouble.", 1),
    ("Absolutely awful. The writing quality was shockingly bad. Full of cliches and predictable tropes. Very disappointed.", 1),
    
    # Neutral reviews (Rating 3)
    ("It was okay. Some parts were good but overall just average. Nothing special or memorable about it.", 3),
    ("Decent read but not something I'd recommend enthusiastically. The story was fine but didn't leave much impact.", 3),
    ("Had potential but fell short. Some interesting ideas but the execution was mediocre at best.", 3),
    ("Middle of the road. Neither great nor terrible. If you have nothing else to read, it'll pass the time.", 3),
]

# Create DataFrame
book_reviews = pd.DataFrame(sample_reviews, columns=['Review', 'Rating'])

print(f"📚 Created {len(book_reviews)} sample book reviews")
print(f"\n📊 Rating distribution:")
print(book_reviews['Rating'].value_counts().sort_index())

# Display sample
book_reviews.head()

📚 Created 19 sample book reviews

📊 Rating distribution:
Rating
1    4
2    3
3    4
4    4
5    4
Name: count, dtype: int64


,Review,Rating
0,This book was absolutely amazing! The plot kep...,5
1,A masterpiece of modern literature. The author...,5
2,Couldn't put it down! Every chapter left me wa...,5
3,One of the best books I've read this year. The...,5
4,Really enjoyed this read. Great character deve...,4


## 2. Preprocess Reviews

Using the **exact same pipeline** as your Amazon Food Review project

In [ ]:
# Initialize preprocessing tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_html(text):
    """Remove HTML tags"""
    if pd.isna(text):
        return ""
    text = re.sub(r'<.*?>', '', str(text))
    return text

def clean_text(text):
    """Clean and preprocess text - MATCHING YOUR ORIGINAL PIPELINE"""
    # Remove HTML
    text = clean_html(text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize (as verbs, matching your code)
    cleaned_tokens = [
        lemmatizer.lemmatize(word, pos='v')
        for word in tokens 
        if word not in stop_words and len(word) > 2
    ]
    
    return ' '.join(cleaned_tokens)

# Apply preprocessing
print("🔧 Preprocessing reviews...")
book_reviews['Cleaned_Review'] = book_reviews['Review'].apply(clean_text)

# Show example
print("\n📝 Example preprocessing:")
print(f"Original: {book_reviews.iloc[0]['Review'][:120]}...")
print(f"Cleaned:  {book_reviews.iloc[0]['Cleaned_Review'][:120]}...")

book_reviews[['Review', 'Cleaned_Review', 'Rating']].head(3)

## 3. Option A: Load Your Saved Model (if you have it)

⚠️ **IMPORTANT**: You need BOTH:
1. The trained model (`amazon_sentiment_final_lr_tfidf.pkl`)
2. The fitted TF-IDF vectorizer (you'll need to save this from your original notebook)

If you don't have the vectorizer saved, skip to **Option B** below.

In [ ]:
# Uncomment and run if you have saved model files:
# model = joblib.load('amazon_sentiment_final_lr_tfidf.pkl')
# vectorizer = joblib.load('tfidf_vectorizer.pkl')  # You need to save this!
# print("✅ Model and vectorizer loaded!")

## 3. Option B: Train a Quick Test Model

Since we may not have your exact saved vectorizer, let's train a simple model to demonstrate the testing process.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create TF-IDF vectorizer (matching your original settings)
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)
)

# For demonstration, we'll train on our sample data
# (In real testing, you'd use your pre-trained food review model)
X_demo = vectorizer.fit_transform(book_reviews['Cleaned_Review'])
y_demo = (book_reviews['Rating'] >= 4).astype(int)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_demo, y_demo)

print("✅ Demo model trained!")
print("⚠️  Note: This is just for demonstration. For real testing, use your pre-trained food model.")

## 4. Make Predictions

In [ ]:
# Vectorize book reviews
X_book = vectorizer.transform(book_reviews['Cleaned_Review'])

# Predict
predictions = model.predict(X_book)
probabilities = model.predict_proba(X_book)

# Add results to dataframe
book_reviews['Predicted_Sentiment'] = predictions
book_reviews['Predicted_Label'] = book_reviews['Predicted_Sentiment'].map({
    1: 'Positive',
    0: 'Negative'
})
book_reviews['Confidence_Positive'] = probabilities[:, 1]
book_reviews['Confidence_Negative'] = probabilities[:, 0]

# Create actual labels (4-5 stars = positive, 1-3 = negative)
book_reviews['Actual_Sentiment'] = (book_reviews['Rating'] >= 4).astype(int)
book_reviews['Actual_Label'] = book_reviews['Actual_Sentiment'].map({
    1: 'Positive',
    0: 'Negative'
})

# Display results
book_reviews[['Review', 'Rating', 'Actual_Label', 'Predicted_Label', 'Confidence_Positive']].head(10)

## 5. Evaluate Performance

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(book_reviews['Actual_Sentiment'], 
                         book_reviews['Predicted_Sentiment'])

print("=" * 70)
print("MODEL PERFORMANCE ON BOOK REVIEWS")
print("=" * 70)
print(f"\n📊 Overall Accuracy: {accuracy:.2%}")
print("\n📈 Classification Report:")
print(classification_report(book_reviews['Actual_Sentiment'], 
                           book_reviews['Predicted_Sentiment'],
                           target_names=['Negative', 'Positive']))

print("\n🔢 Confusion Matrix:")
cm = confusion_matrix(book_reviews['Actual_Sentiment'], 
                     book_reviews['Predicted_Sentiment'])
print(cm)
print("\n   [[True Neg, False Pos],")
print("    [False Neg, True Pos]]")

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix: Book Review Predictions')
plt.ylabel('Actual Sentiment')
plt.xlabel('Predicted Sentiment')
plt.show()

## 6. Domain Transfer Analysis

In [ ]:
print("=" * 70)
print("DOMAIN TRANSFER ANALYSIS")
print("Food Reviews → Book Reviews")
print("=" * 70)

# Find misclassified examples
misclassified = book_reviews[book_reviews['Actual_Sentiment'] != book_reviews['Predicted_Sentiment']]

print(f"\n⚠️  Misclassification Rate: {len(misclassified)/len(book_reviews):.2%}")
print(f"   Total misclassified: {len(misclassified)} out of {len(book_reviews)}")
print(f"\n   - False Positives (predicted positive, actually negative): {len(misclassified[misclassified['Predicted_Sentiment']==1])}")
print(f"   - False Negatives (predicted negative, actually positive): {len(misclassified[misclassified['Predicted_Sentiment']==0])}")

# Analyze confidence scores
print("\n📊 Confidence Analysis:")
print(f"   Average confidence (correct predictions): {book_reviews[book_reviews['Actual_Sentiment']==book_reviews['Predicted_Sentiment']]['Confidence_Positive'].apply(lambda x: max(x, 1-x)).mean():.2%}")
print(f"   Average confidence (incorrect predictions): {misclassified['Confidence_Positive'].apply(lambda x: max(x, 1-x)).mean():.2%}" if len(misclassified) > 0 else "   No misclassifications!")

# Show some misclassified examples
if len(misclassified) > 0:
    print("\n🔍 Example Misclassifications:")
    for idx, row in misclassified.head(3).iterrows():
        print(f"\n   Review: {row['Review'][:100]}...")
        print(f"   Rating: {row['Rating']} | Actual: {row['Actual_Label']} | Predicted: {row['Predicted_Label']}")
        print(f"   Confidence: {row['Confidence_Positive']:.2%}")

## 7. Vocabulary Analysis: Food vs Books

Analyze what words the model learned from food reviews vs what appears in book reviews

In [ ]:
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Get all words from book reviews
all_book_words = ' '.join(book_reviews['Cleaned_Review']).split()
book_vocab = Counter(all_book_words)

print("📚 Most Common Words in Book Reviews:")
print("\nTop 20 words:")
for word, count in book_vocab.most_common(20):
    print(f"   {word:15} {count:3}")

# Book-specific sentiment words
positive_book_words = ['engage', 'character', 'plot', 'write', 'develop', 'story', 'love', 'great', 'excellent']
negative_book_words = ['bore', 'disappoint', 'poor', 'weak', 'terrible', 'awful', 'drag', 'flat']

print("\n✅ Positive Book-Specific Words Found:")
for word in positive_book_words:
    if word in book_vocab:
        print(f"   {word}: {book_vocab[word]} occurrences")

print("\n❌ Negative Book-Specific Words Found:")
for word in negative_book_words:
    if word in book_vocab:
        print(f"   {word}: {book_vocab[word]} occurrences")

## 8. Recommendations for Improvement

Based on testing results, here are suggestions to improve domain transfer:

In [ ]:
print("=" * 70)
print("💡 RECOMMENDATIONS FOR BOOK REVIEW SENTIMENT ANALYSIS")
print("=" * 70)

print("""
1. 🎯 Domain-Specific Training:
   - Collect book review dataset (Goodreads, Amazon Books)
   - Fine-tune your model with book reviews
   - Or use transfer learning: start with food model, retrain on books

2. 📚 Vocabulary Expansion:
   - Add book-specific sentiment lexicon
   - Words like: 'page-turner', 'compelling', 'plot twist', 'character arc'
   - Food words won't help: 'tasty', 'delicious', 'flavor'

3. 🔧 Feature Engineering:
   - Include domain-specific features:
     * Mentions of 'character', 'plot', 'writing'
     * Length of review (detailed reviews often = strong feelings)
     * Presence of comparison words ('better than', 'worse than')

4. 🌐 Try Different Approaches:
   - Use pre-trained BERT/RoBERTa (better at transfer learning)
   - Ensemble: combine food model + book-specific model
   - Zero-shot classification with modern LLMs

5. 📊 Current Model Performance:
""")

if accuracy >= 0.80:
    print(f"   ✅ GOOD ({accuracy:.1%}): Your food model transfers reasonably well!")
    print("      → General sentiment words work across domains")
elif accuracy >= 0.60:
    print(f"   ⚠️  MODERATE ({accuracy:.1%}): Some transfer, but significant room for improvement")
    print("      → Consider fine-tuning on book review data")
else:
    print(f"   ❌ POOR ({accuracy:.1%}): Food vocabulary doesn't transfer well to books")
    print("      → Recommend training a new book-specific model")

print("""
6. 🚀 Next Steps:
   a) Scrape real Goodreads/Amazon book reviews (500-1000 reviews)
   b) Test your ACTUAL food review model on this data
   c) Compare: TF-IDF vs Word2Vec vs BERT
   d) Fine-tune if needed
""")

## 9. Real Web Scraping Code (for when you have internet)

Use this to collect real book reviews:

In [ ]:
# Goodreads Scraper Example (uncomment when you have internet access)

import requests
from bs4 import BeautifulSoup
from time import sleep

def scrape_goodreads_reviews(book_url, max_reviews=50):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    reviews = []
    ratings = []
    
    for page in range(1, max_reviews//10 + 1):
        url = f"{book_url}?page={page}"
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find reviews (selector may need updating)
        review_divs = soup.find_all('div', class_='reviewText')
        rating_divs = soup.find_all('span', class_='staticStars')
        
        for review, rating in zip(review_divs, rating_divs):
            reviews.append(review.get_text(strip=True))
            # Parse rating
            rating_text = rating.get('title', '')
            rating_map = {
                'it was amazing': 5,
                'really liked it': 4,
                'liked it': 3,
                'it was ok': 2,
                'did not like it': 1
            }
            ratings.append(rating_map.get(rating_text.lower(), 3))
        
        sleep(2)  # Be polite to the server
    
    return pd.DataFrame({'Review': reviews, 'Rating': ratings})

# Example usage:
# book_url = "https://www.goodreads.com/book/show/YOUR_BOOK_ID"
# real_reviews = scrape_goodreads_reviews(book_url, max_reviews=100)
"""

print("Scraper code ready! Uncomment and run when you have internet access.")